<a href="https://colab.research.google.com/github/mbaker21231/MicroII-Sandbox/blob/main/Hopenhayn2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Code to simulate Hopenhayn model

A first requirement is a utility function to make a continuous distribution into a grid. Here it is:

In [23]:
#Packages

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

I think it is a good idea to define all the parameters we are using in one place, and basically keep track of them on the basis of whether they are globals or locals, and experimental values.

In [24]:
def tauchen(N, mu, rho, sigma, n_std=4):
    z = np.linspace(mu - n_std * sigma / np.sqrt(1 - rho**2),
                     mu + n_std * sigma / np.sqrt(1 - rho**2), N)
    step = (z[1] - z[0])
    P = np.zeros((N, N))

    for j in range(N):
        for k in range(N):
            if k == 0:
                P[j, k] = norm.cdf((z[k] - rho * z[j] + step / 2) / sigma)
            elif k == N-1:
                P[j, k] = 1 - norm.cdf((z[k] - rho * z[j] - step / 2) / sigma)
            else:
                P[j, k] = (norm.cdf((z[k] - rho * z[j] + step / 2) / sigma) -
                           norm.cdf((z[k] - rho * z[j] - step / 2) / sigma))

    return z, P


A sample of the operations for a 10-point case (too small, but illustrative):

In [25]:
tauchen(5, 0, .5, 1)[1]

array([[1.24106539e-01, 7.51786921e-01, 1.23840537e-01, 2.65998871e-04,
        3.88201826e-09],
       [1.04606677e-02, 4.89539332e-01, 4.89539332e-01, 1.04587379e-02,
        1.92980822e-06],
       [2.66002753e-04, 1.23840537e-01, 7.51786921e-01, 1.23840537e-01,
        2.66002753e-04],
       [1.92980822e-06, 1.04587379e-02, 4.89539332e-01, 4.89539332e-01,
        1.04606677e-02],
       [3.88201827e-09, 2.65998871e-04, 1.23840537e-01, 7.51786921e-01,
        1.24106539e-01]])

Note that we can also use this distribution to recover a cumulative unconditional distribution, which is useful for initial productivity draws:

In [32]:
def init_dist(N, mu, rho, sigma, n_std=4):

    tauch = tauchen(N, mu, rho, sigma, n_std)
    probs = np.sum(tauch[1], axis=0)/np.sum(tauch[1])
    vals = tauch[0]

    return vals, probs

In [35]:
init_dist(10, 0, .5, 1)

(array([-4.61880215, -3.59240167, -2.5660012 , -1.53960072, -0.51320024,
         0.51320024,  1.53960072,  2.5660012 ,  3.59240167,  4.61880215]),
 array([0.00495653, 0.03204551, 0.09999994, 0.16794792, 0.1950501 ,
        0.1950501 , 0.16794792, 0.09999994, 0.03204551, 0.00495653]))

A firm has a flow profit function:
$$
\pi(z) = \tilde z n^\alpha - Wn
$$

where $\tilde z$ is the exponentiated skill level $z$, $\tilde z=e^z$. Flow profits according to:

$$
 \alpha \tilde z n^{\alpha -1}-W \quad \rightarrow\quad n^*(z,w) = \left(\frac{\alpha \tilde z}{W}\right)^\frac{1}{1-\alpha}
$$

In [41]:
def prof_lab(z, W):

  z_tilde = np.exp(z)
  n_sta   = ( alpha * z_tilde / W)**(1/(1-alpha))
  profs   = z_tilde*n_sta**alpha - W*alpha

  return profs, n_sta

In [42]:
ztest = -1,0,1,2
wtest = 1
alpha = .5

In [43]:
prof_lab(ztest,wtest)

(array([-0.43233236,  0.        ,  3.19452805, 26.79907502]),
 array([ 0.03383382,  0.25      ,  1.84726402, 13.64953751]))

In [103]:
def val_fun(z, p, W, max_iter=3000, tol=1e-10):

  v = np.zeros((len(z), 1))

  for i in range(max_iter):

    profs = prof_lab(z, W)[0]
    profs = np.reshape(profs, (len(z), 1))
    vnew = np.maximum( 0, profs - phi_c + (1-beta)* p @ v)
    print(i, abs(vnew-v))
    if np.max(abs(vnew-v)<tol):
      print('here')
      break

  return vnew






In [104]:
profs = prof_lab(zhat, wtest)[0]
profs

array([-4.99999190e-01, -4.99984326e-01, -4.99696629e-01, -4.94128186e-01,
       -3.86349651e-01,  1.69972929e+00,  4.20762789e+01,  8.23573914e+02,
        1.59496449e+04,  3.08718313e+05])

In [105]:
v = np.zeros((len(zhat),1))
phat @ v

array([[0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.]])

In [108]:
phat

array([[2.76726919e-01, 5.36241682e-01, 1.78146262e-01, 8.82652310e-03,
        5.85657570e-05, 4.82083145e-08, 4.71922501e-12, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [3.77201798e-02, 3.45781727e-01, 4.98525975e-01, 1.14141737e-01,
        3.81362179e-03, 1.67497444e-05, 9.02927444e-09, 5.75539616e-13,
        0.00000000e+00, 0.00000000e+00],
       [1.52346614e-03, 6.77156919e-02, 4.30760842e-01, 4.30760842e-01,
        6.77156919e-02, 1.51906018e-03, 4.40441099e-06, 1.55264834e-09,
        6.43929354e-14, 0.00000000e+00],
       [1.67587742e-05, 3.81362179e-03, 1.14141737e-01, 4.98525975e-01,
        3.45781727e-01, 3.71616621e-02, 5.57452999e-04, 1.06444626e-06,
        2.45074405e-10, 6.66133815e-15],
       [4.82130337e-08, 5.85657570e-05, 8.82652310e-03, 1.78146262e-01,
        5.36241682e-01, 2.57690703e-01, 1.88476254e-02, 1.88354218e-04,
        2.36361160e-07, 3.55026009e-11],
       [3.55025863e-11, 2.36361160e-07, 1.88354218e-04, 1.88476254e-02,
   

In [107]:
vvv = val_fun(zhat, phat, wtest)

0 [[0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [6.99729286e-01]
 [4.10762789e+01]
 [8.22573914e+02]
 [1.59486449e+04]
 [3.08717313e+05]]
here


In [97]:
vvv

array([[0.00000000e+00],
       [0.00000000e+00],
       [0.00000000e+00],
       [0.00000000e+00],
       [0.00000000e+00],
       [6.99729286e-01],
       [4.10762789e+01],
       [8.22573914e+02],
       [1.59486449e+04],
       [3.08717313e+05]])

In [65]:
vnew = np.maximum( 0, profs - phi_c + (1-beta)* phat @ v)

In [59]:
profs = prof_lab(zhat, 1)[0]

In [60]:
profs

array([-4.99999190e-01, -4.99984326e-01, -4.99696629e-01, -4.94128186e-01,
       -3.86349651e-01,  1.69972929e+00,  4.20762789e+01,  8.23573914e+02,
        1.59496449e+04,  3.08718313e+05])

In [66]:
vnew

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 6.99729286e-01, 4.10762789e+01, 8.22573914e+02,
        1.59486449e+04, 3.08717313e+05],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 6.99729286e-01, 4.10762789e+01, 8.22573914e+02,
        1.59486449e+04, 3.08717313e+05],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 6.99729286e-01, 4.10762789e+01, 8.22573914e+02,
        1.59486449e+04, 3.08717313e+05],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 6.99729286e-01, 4.10762789e+01, 8.22573914e+02,
        1.59486449e+04, 3.08717313e+05],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 6.99729286e-01, 4.10762789e+01, 8.22573914e+02,
        1.59486449e+04, 3.08717313e+05],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
   